In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<table align="left">
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fgooglemaps-samples%2Finsights-samples%2Fmain%2Fstreet_view_insights%2Ffull_frame%2Ffull_frame_contextual_analysis.ipynb?utm_source=full_frame_street_view_insights_notebooks">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
</table>

# Full-Frame Contextual Analysis with Gemini 3.5 Flash

This notebook demonstrates how to resolve the limitation of classifying small assets within wide-angle Full-Frame Street View images by programmatically cropping the asset using BigQuery bounding box coordinates prior to invoking Gemini 3.5 Flash. It runs a combined scene-understanding analysis: using the Full-Frame image to extract environmental context (road type, weather, surroundings) along with extra object detections, and using the cropped image to classify the target asset itself.

## Install Required Libraries

In [ ]:
!pip install --upgrade google-cloud-bigquery google-genai google-cloud-storage "pillow<11.0.0" matplotlib

## Configuration

**Important**: Replace the placeholder values below with your actual GCP Project ID and Region.

In [ ]:
PROJECT_ID = 'imagery-insights-sandbox'  # @param {type:"string"}
REGION = 'global'      # @param {type:"string"}

# BigQuery Configuration
BIGQUERY_DATASET_ID = 'imagery_insights___us' # @param {type:"string"}
BIGQUERY_TABLE_ID = 'full_frame_observations_latest' # @param {type:"string"}
ASSET_LIMIT = 2 # @param {type:"integer"}
ASSET_TYPE = "ASSET_CLASS_UTILITY_POLE" # @param {type:"string"}
MODEL = "gemini-3.5-flash" # @param {type:"string"}
THINKING_LEVEL = "HIGH" # @param ["MINIMAL", "LOW", "MEDIUM", "HIGH"] {type:"string"}

## Imports and SDK Initialization

In [ ]:
import io
import json
import vertexai
import PIL.Image
import PIL.ImageDraw
import matplotlib.pyplot as plt
from google.cloud import bigquery
from google.cloud import storage
from google import genai
from google.genai import types
from google.genai.types import Content, Part
from pydantic import BaseModel, Field

# Initialize Vertex AI SDK and Gemini Client
vertexai.init(project=PROJECT_ID, location=REGION)
client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

# Define Pydantic Models for structured environmental output
class DetectedObject(BaseModel):
    box_2d: list[int] = Field(description="2D bounding box coordinates [ymin, xmin, ymax, xmax] normalized 0-1000")
    label: str = Field(description="Label of the detected object, e.g. fence, concrete barrier, car, traffic light")

class EnvironmentalAnalysis(BaseModel):
    road_type: str = Field(description="Road type, e.g. Highway, Residential Street")
    surroundings: str = Field(description="Surroundings, e.g. Urban, Suburban, Mountainous")
    lighting_condition: str = Field(description="Lighting/weather condition, e.g. Sunny, Overcast")
    additional_environmental_notes: str = Field(description="Summary of background findings")
    detected_objects: list[DetectedObject] = Field(description="List of key background objects detected with bounding boxes")

## Fetch Observations from BigQuery

We query the BigQuery table to get the GCS URIs of the images and their bounding box coordinates.

In [ ]:
BIGQUERY_SQL_QUERY = f"""
SELECT
  asset_id,
  gcs_uri,
  bbox,
  asset_type
FROM
  `{PROJECT_ID}.{BIGQUERY_DATASET_ID}.{BIGQUERY_TABLE_ID}`
WHERE asset_type = '{ASSET_TYPE}'
LIMIT {ASSET_LIMIT};
"""

try:
    bigquery_client = bigquery.Client(project=PROJECT_ID)
    query_job = bigquery_client.query(BIGQUERY_SQL_QUERY)
    query_response_data = [dict(row) for row in query_job]
    
    observations = []
    for item in query_response_data:
        if item.get("gcs_uri") and item.get("bbox"):
            observations.append({
                "asset_id": item.get("asset_id"),
                "gcs_uri": item.get("gcs_uri"),
                "bbox": item.get("bbox"),
                "asset_type": item.get("asset_type")
            })

    print(f"Successfully fetched {len(observations)} observations.")
    for obs in observations:
        print(obs['gcs_uri'])
except Exception as e:
    print(f"An error occurred while querying BigQuery: {e}")

## Visual Crop and Pipeline Helpers

Define helpers to download images from GCS, crop them programmatically using coordinates, and display comparisons.

In [ ]:
def download_image(gcs_uri: str) -> PIL.Image.Image:
    parts = gcs_uri[5:].split("/", 1)
    bucket_name = parts[0]
    blob_name = parts[1]
    storage_client = storage.Client(project=PROJECT_ID)
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    image_bytes = blob.download_as_bytes()
    return PIL.Image.open(io.BytesIO(image_bytes))

def download_and_crop_image(gcs_uri: str, bbox: dict) -> PIL.Image.Image:
    """
    Downloads the full-frame image from GCS and crops it to the target bounding box coordinates.
    """
    image = download_image(gcs_uri)
    xmin = bbox['lo']['x']
    ymin = bbox['lo']['y']
    xmax = bbox['hi']['x']
    ymax = bbox['hi']['y']
    cropped_image = image.crop((xmin, ymin, xmax, ymax))
    return cropped_image

def display_image_with_bboxes(gcs_uri: str, bq_bbox: dict, bq_label: str = None, gemini_objects: list = None):
    """
    Downloads image, draws BigQuery bbox in red, Gemini bboxes in green, and displays.
    """
    try:
        if not gcs_uri.startswith("gs://"):
            print("Invalid GCS URI")
            return
            
        image = download_image(gcs_uri)
        draw = PIL.ImageDraw.Draw(image)
        width, height = image.size
        
        # 1. Draw BigQuery Bounding Box in Solid Red
        xmin = bq_bbox['lo']['x']
        ymin = bq_bbox['lo']['y']
        xmax = bq_bbox['hi']['x']
        ymax = bq_bbox['hi']['y']
        draw.rectangle([xmin, ymin, xmax, ymax], outline="red", width=12)
        if bq_label:
            draw.text((xmin + 20, ymin + 20), f"[BQ: {bq_label}]", fill="red")
            
        # 2. Draw Gemini Bounding Boxes in Solid Green
        if gemini_objects:
            for obj in gemini_objects:
                box_2d = obj.get("box_2d")
                label = obj.get("label")
-               if len(box_2d) == 4:
+               if box_2d and len(box_2d) == 4:
+                   ymin_g, xmin_g, ymax_g, xmax_g = box_2d
+                   ymin_px = int((ymin_g / 1000) * height)
+                   xmin_px = int((xmin_g / 1000) * width)
+                   ymax_px = int((ymax_g / 1000) * height)
+                   xmax_px = int((xmax_g / 1000) * width)
+                   
+                   draw.rectangle([xmin_px, ymin_px, xmax_px, ymax_px], outline="green", width=8)
+                   draw.text((xmin_px + 20, ymin_px + 20), f"[Gemini: {label}]", fill="green")
+                   
+       plt.figure(figsize=(16, 10))
+       plt.imshow(image)
+       plt.axis('off')
+       plt.show()
+   except Exception as e:
+       print(f"Error displaying image with bounding boxes: {e}")

## Define Pipeline Classification Function

This function supports classifying an image either by its GCS URI or as a PIL Image object, calculating token metrics and API cost dynamically.

In [ ]:
def classify_image_with_gemini(image_input, prompt: str, response_schema=None) -> tuple[str, float, int, int]:
    """
    Classifies an image using Gemini. image_input can be a GCS URI string or a PIL Image object.
    Returns prediction text, calculated cost, prompt tokens, and completion tokens.
    """
    try:
        if isinstance(image_input, str):
            parts = [Part(file_data={'file_uri': image_input, 'mime_type': 'image/jpeg'})]
        else:
            parts = [image_input]
            
        contents = [prompt] + parts
        
        config_args = {
            "thinking_config": types.ThinkingConfig(thinking_level=THINKING_LEVEL)
        }
        if response_schema:
            config_args["response_mime_type"] = "application/json"
            config_args["response_schema"] = response_schema
            
        config = types.GenerateContentConfig(**config_args)
        response = client.models.generate_content(model=MODEL, contents=contents, config=config)
        
        # Calculate cost dynamically from usage metadata
        prompt_tokens = response.usage_metadata.prompt_token_count
        completion_tokens = response.usage_metadata.candidates_token_count
        
        # Pricing for gemini-3.5-flash: Input: $0.000075 / 1k, Output: $0.00030 / 1k
        input_cost = prompt_tokens * (0.000075 / 1000)
        output_cost = completion_tokens * (0.00030 / 1000)
        total_cost = input_cost + output_cost
        
        return response.text, total_cost, prompt_tokens, completion_tokens
    except Exception as e:
        print(f"Error classifying image: {e}")
        return "Classification failed.", 0.0, 0, 0

## Execute Combined Scene Understanding Pipeline

Loop through each observation: crop the image, run environmental analysis on Full-Frame (storing extra bounding boxes), render the overlay drawing, run target asset classification on crop, and print comparison metrics.

In [ ]:
env_prompt = """Analyze the provided wide-angle Street View photo:
Identify the road type, surroundings, lighting conditions, and detect key background/roadside objects (like fences, guardrails, cars, signs, traffic lights).
For every key object you detect, provide its bounding box normalized 0-1000 in the format [ymin, xmin, ymax, xmax].
"""

asset_prompt = """You will be provided with a close-up crop of an asset:
Analyze it and return findings in JSON:
```json
{
  \"pole_condition\": \"OK/Damaged/Other Issues\",
  \"type\": \"<pole_type/sign_type/asset_class>\",
  \"material\": \"<material>\",
  \"transformers\": <number_of_transformers>,
  \"power_lines\": <number_of_power_lines>,
  \"street_lamps\": <number_of_street_lamps>,
  \"junction_boxes\": <number_of_junction_boxes>,
  \"asset_description\": \"<notes>\"
}
```
"""

if 'observations' in locals() and observations:
    for obs in observations:
        uri = obs['gcs_uri']
        bbox = obs['bbox']
        aid = obs['asset_id']
        asset_type = obs['asset_type']
        
        print(f"\n=========================================================================")
        print(f"COMPREHENSIVE PIPELINE RUN FOR OBSERVATION: {uri} (Asset: {aid})")
        print(f"=========================================================================")
        
        # 1. Classify original image for Environmental Context & Bounding Boxes
        print("\n--- Extracting Environmental Context & Gemini Object Detection ---")
        env_res, env_cost, env_in, env_out = classify_image_with_gemini(uri, env_prompt, response_schema=EnvironmentalAnalysis)
        
        # Parse Gemini response
        try:
            analysis = json.loads(env_res)
            detected_objects = analysis.get("detected_objects", [])
        except Exception as e:
            print(f"Error parsing Gemini JSON output: {e}")
            detected_objects = []
            analysis = {}
            
        # 2. Download and crop image programmatically
        print("Downloading and programmatically cropping target asset...")
        cropped_img = download_and_crop_image(uri, bbox)
        
        # 3. Display combined bounding boxes (Red for BQ, Green for Gemini)
        display_image_with_bboxes(uri, bbox, bq_label=asset_type, gemini_objects=detected_objects)
        
        # 4. Classify cropped image for Asset Details (using PIL Image)
        print("\n--- Analyzing Target Asset from Programmatic Crop ---")
        asset_res, asset_cost, asset_in, asset_out = classify_image_with_gemini(cropped_img, asset_prompt)
        
        # 5. Display Pipeline Comparison Report
        print(f"\n---------------------------------------------------------------")
        print(f"COMPREHENSIVE SCENE UNDERSTANDING REPORT")
        print(f"---------------------------------------------------------------")
        print(f"[ENVIRONMENTAL CONTEXT - FULL FRAME]:")
        print(json.dumps(analysis, indent=2))
        print(f"Tokens: Input: {env_in} | Output: {env_out} | Cost: ${env_cost:.6f}")
        print(f"\n[TARGET ASSET DETAILS - PROGRAMMATIC CROP]:\n{asset_res}")
        print(f"Tokens: Input: {asset_in} | Output: {asset_out} | Cost: ${asset_cost:.6f}")
        print(f"---------------------------------------------------------------")
        print(f"Total Scene Processing Cost: ${env_cost + asset_cost:.6f}")
        print(f"---------------------------------------------------------------")
else:
    print("No observations found to run the pipeline on.")